In [1]:
from db_connector import *

========| ('irfan_admin', 'trade_app') |========
*** ✅ SUCCESSFUL CLOUD CONNECTION ⛓️ ***


In [2]:
df = fn_read_data_cloud("gold","bist_master_combined_indicators")

In [ ]:
df = fn_read_data_cloud("gold","ams_master_combined_indicators")

In [ ]:
df = fn_read_data_cloud("gold","nasdaq_master_combined_indicators")

In [ ]:
df = fn_read_data_cloud("gold","nyse_master_combined_indicators")

In [ ]:
df = fn_read_data_cloud("gold","crypto_master_combined_indicators")

In [ ]:
# copy_log_bist_master_combined_indicators
# df = fn_read_data_cloud("test","copy_log_bist_master_combined_indicators")
# df = fn_read_data_cloud("test","log_bist_master_combined_indicators_v2")
# df = fn_read_data_cloud("test","copy_log_nasdaq_master_combined_indicators")
df = fn_read_data_cloud("test","log_nasdaq_master_combined_indicators_v2")



In [ ]:
df

In [3]:
def calc_poc_cluster_bonus(df):
    def cluster_bonus(g):
        g_unique = g.drop_duplicates(subset=["FRVP_HIGHEST_DATE"]).copy()
        count = len(g_unique)

        if count < 2:
            return 0

        pocs = g_unique["FRVP_POC"].dropna().astype(float).values
        if len(pocs) < 2:
            return 0

        spread = (pocs.max() - pocs.min()) / pocs.min()

        if spread <= 0.02:
            if count >= 4:
                return 15
            elif count == 3:
                return 10
            elif count == 2:
                return 5

        return 0

    bonus_df = (
        df.groupby(["EXCHANGE", "SYMBOL"], group_keys=False)
        .apply(lambda g: pd.Series({"score7": cluster_bonus(g)}))
        .reset_index()
    )

    out = df.merge(bonus_df, on=["EXCHANGE", "SYMBOL"], how="left")
    return out["score7"].fillna(0).to_numpy()

In [6]:
import numpy as np
import pandas as pd

# =========================
# HELPERS
# =========================
def to_float(col):
    return (
        col.astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

# =========================
# MFI
# =========================
def calc_mfi_new(df):
    base = (
        np.where(df["MFI"] > df["MFI_YESTERDAY"], 25, 0) +
        np.where(df["MFI"] > df["MFI_12DAY_AVG"], 50, 0) +
        np.where(df["MFI_DIRECTION"] == "Upward", 25, 0)
    )

    scaled = (base / 100) * 70

    # MFI > 80 ise +30 ekle (toplam max 100)
    final = np.where(df["MFI"] > 80, scaled + 30, scaled)

    return final

# =========================
# RSI
# =========================
def calc_rsi_new(df):
    rsi = to_float(df["RSI"])
    rsi_ma = pd.to_numeric(df["RSI_MA"], errors="coerce")
    status = pd.to_numeric(df["RSI_STATUS"], errors="coerce").fillna(0)
    days = pd.to_numeric(df["RSI_CROSS_DAYS_AGO"], errors="coerce").fillna(0)

    score1 = np.where(rsi > rsi_ma, 70, 0)

    days = df["RSI_CROSS_DAYS_AGO"]
    score2 = np.where(
        (df["RSI_STATUS"] == 1) & (days <= 14),
        np.maximum(0, 15 - days),
        0
    )

    score3 = np.where((rsi >= 30) & (rsi <= 70), 15, 0)

    return np.clip(score1 + score2 + score3, 0, 100)

# =========================
# EMA
# =========================
def calc_ema_new(df):
    def ema_score(status, cross, days):
        status = pd.to_numeric(status, errors="coerce").fillna(0)
        cross  = pd.to_numeric(cross, errors="coerce").fillna(0)
        days   = pd.to_numeric(days, errors="coerce").fillna(0)

        return np.where(
            (status == 1) & (cross == 1),
            np.maximum(0, 3 - (days / 10)),
            0
        )

    total = (
        ema_score(df["EMA_STATUS_5_20"], df["EMA_CROSS_5_20"], df["EMA_DAYS_SINCE_CROSS_5_20"]) +
        ema_score(df["EMA_STATUS_3_20"], df["EMA_CROSS_3_20"], df["EMA_DAYS_SINCE_CROSS_3_20"]) +
        ema_score(df["EMA_STATUS_3_14"], df["EMA_CROSS_3_14"], df["EMA_DAYS_SINCE_CROSS_3_14"])
    )

    return np.clip((total / 9) * 100, 0, 100)

# =========================
# VOLUME
# =========================
def calc_volume_new(df):
    vol_last = to_float(df["VOL_LASTDAY"])
    vol_yest = to_float(df["VOL_YESTERDAY"])
    vol_5 = to_float(df["VOL_AVG_5DAY"])
    vol_10 = to_float(df["VOL_AVG_10DAY"])
    vol_20 = to_float(df["VOL_AVG_20DAY"])

    return (
        np.where(vol_last > vol_yest, 25, 0) +
        np.where(vol_last > vol_5, 25, 0) +
        np.where(vol_5 > vol_10, 25, 0) +
        np.where(vol_5 > vol_20, 25, 0)
    )

# =========================
# VWAP
# =========================
def calc_vwap_new(df):
    close = df["FRVP_LATEST_CLOSE_VALUE"]
    vwap = df["VWAP"]

    dist = np.abs(close - vwap) / vwap

    return np.clip(
        np.where(dist <= 0.05, 0,
                 np.where(dist >= 0.10, 100,
                          ((dist - 0.05) / 0.05) * 100)),
        0, 100
    )

# =========================
# POC FRVP
# =========================
import numpy as np
import pandas as pd

def calc_poc_frvp_new(df, log_file="poc_log.txt"):

    close = df["FRVP_LATEST_CLOSE_VALUE"].astype(float)
    poc = df["FRVP_POC"].astype(float)
    val = df["FRVP_VAL"].astype(float)
    vah = df["FRVP_VAH"].astype(float)

    # =========================
    # VALUE AREA WIDTH
    # =========================
    value_range = vah - val

    # =========================
    # 1 → CLOSE vs POC (10 puan)
    # =========================
    dist_close_poc = np.abs(close - poc)

    score1 = np.where(
        dist_close_poc <= (value_range * 0.05),
        10,
        np.where(
            dist_close_poc <= (value_range * 0.80),
            (1 - (dist_close_poc - value_range * 0.05) / (value_range * 0.75)) * 10,
            0
        )
    )

    score1 = np.clip(score1, 0, 10)
    df["score1"] = score1

    # =========================
    # 2 → CLUSTER BONUS (15 puan)
    # =========================
    score2 = calc_poc_cluster_bonus(df)
    df["score2"] = score2

    # =========================
    # 3 → GREEN (5 puan)
    # =========================
    score3 = np.where(df["BS_BAR_STATUS"] == "GREEN", 5, 0)
    df["score3"] = score3

    # =========================
    # 4 → CONTACTLESS (5 puan)
    # =========================
    score4 = np.where(
        (df["BS_OPEN_PRICE"].astype(float) > poc) &
        (df["BS_BAR_STATUS"] == "GREEN"),
        5,
        0
    )
    df["score4"] = score4

    # =========================
    # 5 → POC vs VAL (5 puan)
    # =========================
    dist_poc_val = np.abs(poc - val)

    score5 = np.where(
        dist_poc_val <= (value_range * 0.10),
        5,
        np.where(
            dist_poc_val <= (value_range * 0.40),
            (1 - (dist_poc_val - value_range * 0.10) / (value_range * 0.30)) * 5,
            0
        )
    )

    score5 = np.clip(score5, 0, 5)
    df["score5"] = score5

    # =========================
    # TOTAL (max = 40)
    # =========================
    total = score1 + score2 + score3 + score4 + score5
    df["total_score"] = total

    final_score = np.clip((total / 40) * 100, 0, 100)
    df["poc_frvp_status"] = final_score

#     # =========================
#     # LOG
#     # =========================
#     with open(log_file, "w", encoding="utf-8") as f:

#         for i in range(len(df)):

#             f.write(f"""
# SYMBOL: {df.iloc[i]["SYMBOL"]}
# PERIOD: {df.iloc[i]["FRVP_PERIOD_TYPE"]}
# HIGHEST_DATE: {df.iloc[i]["FRVP_HIGHEST_DATE"]}
# GUN: {df.iloc[i]["CREATED_DAY"]}

# VAL: {val.iloc[i]:.2f} | POC: {poc.iloc[i]:.2f} | VAH: {vah.iloc[i]:.2f}
# CLOSE: {close.iloc[i]:.2f}

# score1 (close→poc): {score1[i]:.2f}
# score2 (cluster): {score2[i]}
# score3 (green): {score3[i]}
# score4 (contactless): {score4[i]}
# score5 (poc→val): {score5[i]:.2f}

# TOTAL: {total[i]:.2f} → FINAL: {final_score[i]:.2f}
# -------------------------------------------
# """)

    return final_score

# =========================
# TRADE LEVELS
# =========================

def calculate_trade_levels(df):

    close = df["FRVP_LATEST_CLOSE_VALUE"]
    df["entry_price"] = close * 1.005

    pivot = df["PIVOT"] if "PIVOT" in df.columns else pd.Series(np.nan, index=df.index)
    r2 = df["R2"] if "R2" in df.columns else pd.Series(np.nan, index=df.index)

    def choose_target(row):

        c = row["FRVP_LATEST_CLOSE_VALUE"]

        if c < row["FRVP_POC"]:
            return row["VWAP"]

        elif c < row["VWAP"]:
            return row["VWAP"]

        elif c < row["FRVP_VAH"]:
            return row["FRVP_VAH"]

        elif pd.notna(pivot.loc[row.name]) and c < pivot.loc[row.name]:
            return pivot.loc[row.name]

        elif pd.notna(r2.loc[row.name]):
            return r2.loc[row.name]

        else:
            return row["FRVP_VAH"]

    df["target_price"] = df.apply(choose_target, axis=1)

    df["target_pct"] = ((df["target_price"] - df["entry_price"]) / df["entry_price"]) * 100
    df["risk_pct"] = ((df["entry_price"] - df["stop_loss"]) / df["entry_price"]) * 100
    df["rr_ratio"] = df["target_pct"] / df["risk_pct"]

    df["pivot_display"] = np.where(
        pivot.isna(),
        "N/A",
        pivot
    )

    return df

# =========================
# MASTER FUNCTION
# =========================

def calculate_scores(df):

    df["poc_frvp_status"] = calc_poc_frvp_new(df)
    df["vwap_status"] = calc_vwap_new(df)
    df["ema_status"] = calc_ema_new(df)
    df["rsi_status"] = calc_rsi_new(df)
    df["mfi_status"] = calc_mfi_new(df)
    df["vol_status"] = calc_volume_new(df)

    # =========================
    # MASTER SCORE
    # =========================
    df["master_score"] = (
        df["poc_frvp_status"] * 0.62 +
        df["vwap_status"] * 0.02 +
        df["ema_status"] * 0.09 +
        df["rsi_status"] * 0.07 +
        df["mfi_status"] * 0.10 +
        df["vol_status"] * 0.10
    )

    df["master_score"] = np.clip(df["master_score"], 0, 100)

    # =========================
    # WATCHLIST
    # =========================
    df["watchlist"] = np.where(
        df["master_score"] >= 70, "BUY",
        np.where(df["master_score"] >= 50, "WATCH", "AVOID")
    )
    df["entry_price"] = df["FRVP_LATEST_CLOSE_VALUE"] * 1.005
    

    # =========================
    # STOP LOSS (ENTRY’den sonra!)
    # =========================
    df["stop_loss"] = df["entry_price"] * 0.97

    # =========================
    # TRADE LEVELS (ENTRY burada oluşuyor)
    # =========================
    df = calculate_trade_levels(df)

    return df

In [7]:
scores = calculate_scores(df)

# --- TRIAGE KISMI----

In [8]:
def calculate_triage_selection(df):

    # df_filtered = df[df["BS_CLOSE_PRICE"] > df["FRVP_POC"]].copy()
    df_filtered = df[
    (df["BS_CLOSE_PRICE"] > df["FRVP_POC"])].copy()

    if df_filtered.empty:
        return pd.DataFrame()

    grouped = df_filtered.groupby(["EXCHANGE", "SYMBOL"])

    results = []

    for (exchange, symbol), g in grouped:

        # =========================
        # UNIQUE CLUSTER COUNT
        # =========================
        g_unique = g.drop_duplicates(subset=["FRVP_HIGHEST_DATE"])
        count = len(g_unique)

        # =========================
        # SCORE
        # =========================
        avg_score = g_unique["master_score"].mean()

        # =========================
        # POC CLUSTER ANALYSIS
        # =========================
        pocs = g_unique["FRVP_POC"].values

        poc_counts = g["FRVP_POC"].value_counts()

        max_freq = poc_counts.max()

        dominant_pocs = poc_counts[poc_counts == max_freq].index

        avg_poc = np.mean(dominant_pocs)

        # =========================
        # TRADE LEVELS (UPDATED)
        # =========================

        last_close = g["BS_CLOSE_PRICE"].iloc[-1]

        # STOP LOSS → %3 aşağı
        stop_loss_price = last_close * (1 - 0.03)

        # TARGET → mevcut avg_target yerine relative hesap
        avg_target = g["target_price"].mean()

        target_pct = ((avg_target - last_close) / last_close) * 100

        triage_day = pd.to_datetime(g["CREATED_DAY"].iloc[0], dayfirst=True).strftime("%Y-%m-%d")

        results.append({
            "EXCHANGE": exchange,
            "SYMBOL": symbol,
            "triage_entry_day": triage_day,

            "triage_score": round(min(100, avg_score), 2),
            "valid_cluster_count": count,

            "avg_poc": f"{avg_poc:.2f}".replace(".", ","),

            # ✅ yeni yapı
            "stop_loss": f"{stop_loss_price:.2f}".replace(".", ","),
            "target_price": f"{avg_target:.2f}".replace(".", ","),

            # ✅ yüzde artık dinamik
            "target_%": f"{target_pct:.2f}%".replace(".", ",")
        })

    result_df = pd.DataFrame(results)

    result_df = result_df.sort_values("triage_score", ascending=False)

    return result_df


In [9]:
def log_poc_detail(df, log_file="poc_log.txt"):

    with open(log_file, "w", encoding="utf-8") as f:

        grouped = df.groupby(["EXCHANGE", "SYMBOL", "CREATED_DAY"])

        for (exchange, symbol, day), g in grouped:

            g = g.sort_values("FRVP_HIGHEST_DATE", ascending=False)

            # triage score (tek)
            triage_score = g["TRIAGE_SCORE"].iloc[0] if "TRIAGE_SCORE" in g else "N/A"

            for i in range(len(g)):

                r = g.iloc[i]

                f.write(f"""
SYMBOL: {r["SYMBOL"]}
PERIOD: {r["FRVP_PERIOD_TYPE"]}
HIGHEST_DATE: {r["FRVP_HIGHEST_DATE"]}
GUN: {r["CREATED_DAY"]}

VAL: {r["FRVP_VAL"]:.2f} | POC: {r["FRVP_POC"]:.2f} | VAH: {r["FRVP_VAH"]:.2f}
VWAP: {r["VWAP"]:.2f}
CLOSE: {r["FRVP_LATEST_CLOSE_VALUE"]:.2f}

score1 (close→poc): {r["score1"]:.2f}
score2 (cluster): {r["score2"]:.2f}
score3 (green): {r["score3"]}
score4 (contactless): {r["score4"]}
score5 (poc→val): {r["score5"]:.2f}

TOTAL: {r["total_score"]:.2f} → FINAL: {r["poc_frvp_status"]:.2f}

MASTER SCORE: {r["master_score"]:.2f}
TRIAGE SCORE: {triage_score}

-------------------------------------------
""")

In [10]:
def log_technical_summary(df, log_file="tech_log.txt"):

    with open(log_file, "w", encoding="utf-8") as f:

        grouped = df.groupby(["EXCHANGE", "SYMBOL", "CREATED_DAY"])

        for (exchange, symbol, day), g in grouped:

            g = g.sort_values("FRVP_HIGHEST_DATE", ascending=False)

            g_unique = g.drop_duplicates(subset=["FRVP_HIGHEST_DATE"])
            cluster_count = len(g_unique)

            # =========================
            # POC + VWAP birlikte
            # =========================
            poc_lines = []
            for _, row in g_unique.iterrows():
                poc_lines.append(
                    f"{row['FRVP_PERIOD_TYPE']} | {row['FRVP_HIGHEST_DATE']} "
                    f"| POC:{row['FRVP_POC']:.2f} | VWAP:{row['VWAP']:.2f}"
                )

            poc_text = "\n".join(poc_lines)

            r = g.iloc[0]

            # =========================
            # EMA açıklama
            # =========================
            def ema_text(status, days):
                if status == 1:
                    return f"yukarı kesişim, {int(days)} gündür önde"
                elif status == -1:
                    return f"aşağı kesişim, {int(days)} gündür altta"
                else:
                    return "kesişim yok"

            ema_5_20 = ema_text(r["EMA_STATUS_5_20"], r["EMA_DAYS_SINCE_CROSS_5_20"])
            ema_3_20 = ema_text(r["EMA_STATUS_3_20"], r["EMA_DAYS_SINCE_CROSS_3_20"])
            ema_3_14 = ema_text(r["EMA_STATUS_3_14"], r["EMA_DAYS_SINCE_CROSS_3_14"])

            f.write(f"""
===================== {symbol} =====================

📅 Gün: {day}
📊 Cluster Sayısı: {cluster_count}

📍 POC + VWAP CLUSTERS:
{poc_text}

💰 PRICE
Close:{r["BS_CLOSE_PRICE"]:.2f} | Open:{r["BS_OPEN_PRICE"]:.2f}

📉 EMA
EMA3:{r["EMA3"]:.2f} EMA5:{r["EMA5"]:.2f} EMA14:{r["EMA14"]:.2f} EMA20:{r["EMA20"]:.2f}
5-20: {ema_5_20}
3-20: {ema_3_20}
3-14: {ema_3_14}

📈 RSI
RSI:{r["RSI"]:.2f} | MA:{r["RSI_MA"]:.2f} | Status:{r["RSI_STATUS"]}

💸 MFI
Bugün:{r["MFI"]:.2f} | Dün:{r["MFI_YESTERDAY"]:.2f} | 12g Ort:{r["MFI_12DAY_AVG"]:.2f} | Yön:{r["MFI_DIRECTION"]}

📦 VOLUME
Bugün:{r["VOL_LASTDAY"]} | Dün:{r["VOL_YESTERDAY"]} 
5g:{r["VOL_AVG_5DAY"]} | 10g:{r["VOL_AVG_10DAY"]} | 20g:{r["VOL_AVG_20DAY"]}
Durum:{r["VOL_STATUS"]}

📌 PIVOT
P:{r["PIVOT"]:.2f} R1:{r["PVT_R1"]:.2f} R2:{r["PVT_R2"]:.2f} R3:{r["PVT_R3"]:.2f}
S1:{r["PVT_S1"]:.2f} S2:{r["PVT_S2"]:.2f} S3:{r["PVT_S3"]:.2f}

🧠 SCORE
POC:{r["poc_frvp_status"]:.2f} VWAP:{r["vwap_status"]:.2f} EMA:{r["ema_status"]:.2f}
RSI:{r["rsi_status"]:.2f} MFI:{r["mfi_status"]:.2f} VOL:{r["vol_status"]:.2f}

⭐ MASTER: {r["master_score"]:.2f}

------------------------------------------------------------
""")

In [13]:
def build_final_triage_df(df):

    triage_df = calculate_triage_selection(df)

    if triage_df.empty:
        return pd.DataFrame()

    # rank
    triage_df["rank"] = (
        triage_df
        .groupby("triage_entry_day")["triage_score"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # merge key
    df["triage_entry_day"] = pd.to_datetime(df["CREATED_DAY"], dayfirst=True).dt.strftime("%Y-%m-%d")

    merged = df.merge(
        triage_df,
        on=["EXCHANGE", "SYMBOL", "triage_entry_day"],
        how="left"
    )

    # =========================
    # YENİ KOLONLAR
    # =========================
    merged["MASTER_SCORE"] = merged["master_score"].round(2)

    merged["RSI_STATUS_TXT"] = merged["rsi_status"]
    merged["MFI_STATUS_TXT"] = merged["mfi_status"]
    merged["VOL_STATUS_TXT"] = merged["vol_status"]

    merged["STOP_LOSS_PERC"] = "-3,00%"
    merged["TARGET_PERC"] = merged.get("target_%", None)

    merged["RUNTIME"] = np.nan

    # rename triage
    merged.rename(columns={
        "triage_score": "TRIAGE_SCORE",
        "valid_cluster_count": "VALID_CLUSTER_COUNT",
        "avg_poc": "AVG_POC",
        "stop_loss_y": "STOP_LOSS",
        "target_price_y": "TARGET_PRICE",
        "rank": "RANK",
        "triage_entry_day": "TRIAGE_ENTRY_DAY"
    }, inplace=True)
    # =========================
    # 🔥 LOG (EN SON)
    # =========================
    # log_poc_detail(df)
    log_technical_summary(df)


    # =========================
    # FINAL KOLONLAR (senin verdiğin sıraya göre)
    # =========================
    final_cols = [
        "EXCHANGE","SYMBOL","FRVP_INTERVAL","FRVP_PERIOD_TYPE",
        "FRVP_HIGHEST_DATE","FRVP_HIGHEST_VALUE",
        "FRVP_ROW_COUNT_AFTER_HIGHEST","FRVP_DAY_COUNT_AFTER_HIGHEST",
        "FRVP_LATEST_CLOSE_VALUE","FRVP_POC","FRVP_VAL","FRVP_VAH",
        "BS_OPEN_PRICE","BS_CLOSE_PRICE","BS_DIFFER","BS_PERC","BS_BAR_STATUS",
        "RAW_END_DATE","BRONZE_END_DATE","SILVER_END_DATE","SILVER_CONVERTED_END_DATE","EXPECTED_END_DATE",
        "EMA_TIMESTAMP","EMA_END_DATE","EMA3","EMA5","EMA14","EMA20",
        "EMA_STATUS_5_20","EMA_CROSS_5_20",
        "EMA_STATUS_3_20","EMA_CROSS_3_20",
        "EMA_STATUS_3_14","EMA_CROSS_3_14",
        "EMA_DAYS_SINCE_CROSS_5_20","EMA_DAYS_SINCE_CROSS_3_20","EMA_DAYS_SINCE_CROSS_3_14",
        "RSI","RSI_MA","RSI_STATUS","RSI_CROSS","RSI_CROSS_DAYS_AGO",
        "MFI","MFI_YESTERDAY","MFI_12DAY_AVG","MFI_DIRECTION",
        "VWAP_HIGHEST_VALUE","VWAP_HIGHEST_TIMESTAMP","VWAP",
        "VOL_AVG_5DAY","VOL_AVG_10DAY","VOL_AVG_20DAY",
        "VOL_YESTERDAY","VOL_LASTDAY","VOL_STATUS",
        "PVT_START_DATE","PVT_END_DATE","PVT_YEAR",
        "PIVOT","PVT_R1","PVT_R2","PVT_R3","PVT_R4","PVT_R5",
        "PVT_S1","PVT_S2","PVT_S3","PVT_S4","PVT_S5","PVT_STATUS",

        "poc_frvp_status","vwap_status","ema_status",
        "RSI_STATUS_TXT","MFI_STATUS_TXT","VOL_STATUS_TXT",

        "MASTER_SCORE",

        "TRIAGE_ENTRY_DAY","TRIAGE_SCORE","RANK",
        "VALID_CLUSTER_COUNT","AVG_POC",
        "STOP_LOSS","TARGET_PRICE","STOP_LOSS_PERC","TARGET_PERC",

        "CREATED_AT","CREATED_DAY","RUNTIME"
    ]
    final_cols = [c for c in final_cols if c in merged.columns]

     

    return merged[final_cols]

In [ ]:
son_hali = build_final_triage_df(scores)

In [ ]:
print(son_hali.to_clipboard(index=False))

BACK TEST (SADECE YASIN ICIN)

In [ ]:
df.head(10)

In [ ]:
# hourly ilgili olani al
# df_hourly = fn_read_data_cloud("raw","bist_hourly_archive")
df_hourly = fn_read_data_cloud("raw","nasdaq_hourly_archive")

In [ ]:
import pandas as pd
import numpy as np


RESULT_COLUMNS = [
    "expected_entry_price",
    "entry_time",
    "day1_chart",
    "calculated_stop_loss",
    "close_position_day",
    "position_duration",
    "close_position_price",
    "return_%",
    "max_price_seen",
    "max_price_day",
    "max_return_%",
    "exit_reason",
]


def _to_float_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", ".", regex=False)
         .str.replace("%", "", regex=False)
         .str.strip(),
        errors="coerce"
    )


def prepare_signals(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip()
    col_map = {
        "TRIAGE_ENTRY_DAY": "triage_entry_day",
        "TARGET_PRICE": "target_price"
    }

    for old, new in col_map.items():
        if old in df.columns and new not in df.columns:
            df[new] = df[old]

    required_cols = ["SYMBOL", "triage_entry_day", "target_price"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Signal dataframe missing columns: {missing}")

    df["SYMBOL"] = df["SYMBOL"].astype(str).str.strip()
    df["triage_entry_day"] = pd.to_datetime(df["triage_entry_day"], errors="coerce")
    df["target_price"] = _to_float_series(df["target_price"])

    return df


def prepare_hourly(df_hourly: pd.DataFrame) -> pd.DataFrame:
    dfh = df_hourly.copy()
    dfh.columns = dfh.columns.str.strip()

    required_cols = ["SYMBOL", "TIMESTAMP", "OPEN", "HIGH", "LOW", "CLOSE"]
    missing = [c for c in required_cols if c not in dfh.columns]
    if missing:
        raise ValueError(f"Hourly dataframe missing columns: {missing}")

    dfh["SYMBOL"] = dfh["SYMBOL"].astype(str).str.strip()
    dfh["TIMESTAMP"] = pd.to_datetime(dfh["TIMESTAMP"], errors="coerce")

    for col in ["OPEN", "HIGH", "LOW", "CLOSE"]:
        dfh[col] = _to_float_series(dfh[col])

    dfh = dfh.dropna(subset=["SYMBOL", "TIMESTAMP", "OPEN", "HIGH", "LOW", "CLOSE"]).copy()
    dfh["date"] = dfh["TIMESTAMP"].dt.date
    dfh["hour"] = dfh["TIMESTAMP"].dt.hour

    dfh = dfh.sort_values(["SYMBOL", "TIMESTAMP"]).reset_index(drop=True)
    return dfh


def empty_result(reason=np.nan) -> dict:
    return {
        "expected_entry_price": np.nan,
        "entry_time": pd.NaT,
        "calculated_stop_loss": np.nan,
        "close_position_day": pd.NaT,
        "position_duration": np.nan,
        "close_position_price": np.nan,
        "return_%": np.nan,
        "max_price_seen": np.nan,
        "max_price_day": pd.NaT,
        "max_return_%": np.nan,
        "exit_reason": reason,
    }

def evaluate_one_position(
    row: pd.Series,
    hourly_by_symbol: dict,
    trading_days_by_symbol: dict,
    stop_loss_pct: float = 0.03,
):

    symbol = row["SYMBOL"]
    signal_day = row["triage_entry_day"]

    if pd.isna(signal_day):
        return empty_result("missing_signal_data")

    signal_day = signal_day.date()

    sym_df = hourly_by_symbol.get(symbol)
    sym_days = trading_days_by_symbol.get(symbol)

    if sym_df is None or sym_df.empty or not sym_days:
        return empty_result("missing_hourly_data")

    # =========================
    # TRIAGE DAY CLOSE
    # =========================
    triage_day_df = sym_df[sym_df["date"] == signal_day]

    if triage_day_df.empty:
        return empty_result("missing_triage_day")

    triage_close = float(triage_day_df.iloc[-1]["CLOSE"])

    # =========================
    # FUTURE DAYS
    # =========================
    future_days = [d for d in sym_days if d > signal_day]
    if not future_days:
        return empty_result("no_future_trading_day")

    # Day+1
    t1 = future_days[0]

    day_df = sym_df.loc[sym_df["date"] == t1].sort_values("TIMESTAMP").reset_index(drop=True)
    if len(day_df) < 2:
        return empty_result("insufficient_day1_bars")

    second_candle = day_df.iloc[1]

    entry_price = float(second_candle["CLOSE"])
    entry_time = second_candle["TIMESTAMP"]

    # =========================
    # ✅ YENİ day1_chart (matematiksel)
    # =========================
    day1_chart = "GREEN" if entry_price > triage_close else "RED"

    # =========================
    # STOP LOSS
    # =========================
    stop_loss = entry_price * (1 - stop_loss_pct)

    # =========================
    # 7 GÜN
    # =========================
    allowed_days = future_days[:7]

    df_future = sym_df.loc[
        (sym_df["TIMESTAMP"] > entry_time) &
        (sym_df["date"].isin(allowed_days))
    ].sort_values("TIMESTAMP").reset_index(drop=True)

    if df_future.empty:
        return empty_result("no_future_bars")

    close_time = None
    close_price = None
    exit_reason = None

    max_price = entry_price
    max_time = entry_time

    for _, bar in df_future.iterrows():

        low = float(bar["LOW"])
        high = float(bar["HIGH"])
        ts = bar["TIMESTAMP"]

        if low <= stop_loss:
            close_time = ts
            close_price = stop_loss
            exit_reason = "trailing_stop"
            break

        if high > max_price:
            max_price = high
            max_time = ts

        new_stop = high * (1 - stop_loss_pct)

        if new_stop > stop_loss:
            stop_loss = new_stop

    if close_time is None:
        last_bar = df_future.iloc[-1]
        close_time = last_bar["TIMESTAMP"]
        close_price = float(last_bar["CLOSE"])
        exit_reason = "timeout_close"

    duration_days = (close_time.date() - entry_time.date()).days

    return_pct = ((close_price / entry_price) - 1.0) * 100
    max_return_pct = ((max_price / entry_price) - 1.0) * 100

    return {
        "expected_entry_price": round(entry_price, 4),
        "entry_time": entry_time,
        "day1_chart": day1_chart, 
        "calculated_stop_loss": round(stop_loss, 4),
        "close_position_day": close_time,
        "position_duration": f"{duration_days} days",
        "close_position_price": round(close_price, 4),
        "return_%": round(return_pct, 2),
        "max_price_seen": round(max_price, 4),
        "max_price_day": max_time,
        "max_return_%": round(max_return_pct, 2),
        "exit_reason": exit_reason,
    }


def build_hourly_cache(df_hourly: pd.DataFrame):
    hourly_by_symbol = {}
    trading_days_by_symbol = {}

    for symbol, g in df_hourly.groupby("SYMBOL", sort=False):
        gs = g.sort_values("TIMESTAMP").reset_index(drop=True).copy()
        hourly_by_symbol[symbol] = gs
        trading_days_by_symbol[symbol] = sorted(gs["date"].unique().tolist())

    return hourly_by_symbol, trading_days_by_symbol


def run_backtest(
    signals_df: pd.DataFrame,
    hourly_df: pd.DataFrame,
    stop_loss_pct: float = 0.03,
) -> pd.DataFrame:
    signals = prepare_signals(signals_df)
    hourly = prepare_hourly(hourly_df)

    hourly_by_symbol, trading_days_by_symbol = build_hourly_cache(hourly)

    results = [
        evaluate_one_position(
            row,
            hourly_by_symbol,
            trading_days_by_symbol,
            stop_loss_pct=stop_loss_pct,
        )
        for _, row in signals.iterrows()
    ]

    results_df = pd.DataFrame(results, index=signals.index)
    final_df = pd.concat([signals, results_df], axis=1)

    ordered_cols = list(signals.columns) + RESULT_COLUMNS
    ordered_cols = [c for c in ordered_cols if c in final_df.columns]

    return final_df[ordered_cols]

In [ ]:
def apply_backtest_to_triage(df, df_hourly):

    df = df.copy()

    # =========================
    # SADECE TOP 10
    # =========================
    df_bt = df[df["RANK"] <= 10].copy()

    if df_bt.empty:
        return df

    # =========================
    # BACKTEST ÇALIŞTIR
    # =========================
    bt_results = run_backtest(df_bt, df_hourly)

    # =========================
    # SADECE ENTRY OLANLAR
    # =========================
    bt_results = bt_results[bt_results["entry_time"].notna()]

    # =========================
    # SONUÇ KOLONLARI
    # =========================
    for col in RESULT_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    # =========================
    # MERGE (index bazlı)
    # =========================
    df[RESULT_COLUMNS] = df[RESULT_COLUMNS].astype("object")

    # 🔥 update yerine assign
    df.loc[bt_results.index, RESULT_COLUMNS] = bt_results[RESULT_COLUMNS]

    return df

In [ ]:
son_df = apply_backtest_to_triage(son_hali, df_hourly)

In [ ]:
def get_top3_per_day(df):

    df_filtered = df[df["entry_time"].notna()].copy()

    if df_filtered.empty:
        return pd.DataFrame()

    df_filtered["entry_day"] = pd.to_datetime(df_filtered["entry_time"]).dt.date

    sort_col = "triage_score" if "triage_score" in df_filtered.columns else "return_%"

    df_filtered = df_filtered.sort_values(
        ["entry_day", sort_col],
        ascending=[True, False]
    )

    # önce rank'e göre sırala
    df_filtered = df_filtered.sort_values(
        ["entry_day", "RANK"],
        ascending=[True, True]
    )

    df_filtered = df_filtered.drop_duplicates(
        subset=["EXCHANGE", "SYMBOL", "entry_time"]
    )

    final_cols = [
        "EXCHANGE","SYMBOL","CREATED_DAY","RANK",
        "expected_entry_price","entry_time","day1_chart","calculated_stop_loss",
        "close_position_day","position_duration","close_position_price",
        "return_%","max_price_seen","max_price_day","max_return_%",
        "exit_reason"
    ]

    final_cols = [c for c in final_cols if c in df_filtered.columns]

    return df_filtered[final_cols].reset_index(drop=True)

In [ ]:
son_3 = get_top3_per_day(son_df)

In [ ]:
print(son_3.to_clipboard(index=False))

In [ ]:
def calculate_7_days_output(df, df_hourly):

    df = df.copy()

    # =========================
    # GÜN + SYMBOL TEKİL (EN İYİ RANK)
    # =========================
    df["triage_day"] = pd.to_datetime(df["CREATED_DAY"], dayfirst=True).dt.date

    df = (
        df.sort_values(["triage_day", "RANK"])
        .drop_duplicates(subset=["triage_day", "SYMBOL"])
    )

    # =========================
    # HER GÜN İLK 10
    # =========================
    df = df.groupby("triage_day", group_keys=False).head(10)

    # =========================
    # HOURLY PREP
    # =========================
    dfh = df_hourly.copy()
    dfh["TIMESTAMP"] = pd.to_datetime(dfh["TIMESTAMP"])
    dfh["date"] = dfh["TIMESTAMP"].dt.date
    dfh = dfh.sort_values(["SYMBOL", "TIMESTAMP"])

    results = []

    for _, row in df.iterrows():

        symbol = row["SYMBOL"]
        triage_day = row["triage_day"]

        sym_df = dfh[dfh["SYMBOL"] == symbol]

        if sym_df.empty:
            continue

        trading_days = sorted(sym_df["date"].unique())
        future_days = [d for d in trading_days if d > triage_day]

        if len(future_days) < 1:
            continue

        # =========================
        # DAY+1
        # =========================
        day1 = future_days[0]
        day1_df = sym_df[sym_df["date"] == day1].sort_values("TIMESTAMP")

        if len(day1_df) < 1:
            continue

        first_candle = day1_df.iloc[0]

        # triage day close'u al
        triage_day_close_df = sym_df[sym_df["date"] == triage_day]

        if triage_day_close_df.empty:
            continue

        triage_close = float(triage_day_close_df.iloc[-1]["CLOSE"])

        # entry (day+1 first candle)
        entry_price = float(first_candle["CLOSE"])
        entry_time = first_candle["TIMESTAMP"]

        # ✅ yeni mantık
        day1_chart = "GREEN" if entry_price > triage_close else "RED"

        # =========================
        # SONRAKİ 7 GÜN
        # =========================
        next_days = future_days[:7]

        df_future = sym_df[
            (sym_df["date"].isin(next_days)) &
            (sym_df["TIMESTAMP"] >= entry_time)
        ]

        if df_future.empty:
            continue

        max_price = df_future["HIGH"].max()
        min_price = df_future["LOW"].min()

        # 7.gün kapanış
        day7 = next_days[-1]
        day7_df = sym_df[sym_df["date"] == day7]

        close_7 = float(day7_df.iloc[-1]["CLOSE"]) if not day7_df.empty else None

        # =========================
        # %
        # =========================
        max_pct = ((max_price / entry_price) - 1) * 100 if max_price else None
        min_pct = ((min_price / entry_price) - 1) * 100 if min_price else None
        close7_pct = ((close_7 / entry_price) - 1) * 100 if close_7 else None

        results.append({
            "EXCHANGE": row["EXCHANGE"],
            "SYMBOL": symbol,
            "CREATED_DAY": triage_day,
            "RANK": row["RANK"],

            "entry_price": round(entry_price, 2),
            "entry_time": entry_time,
            "day1_chart": day1_chart,

            "max_price_7d": round(max_price, 2),
            "max_pct_7d": round(max_pct, 2) if max_pct else None,

            "min_price_7d": round(min_price, 2),
            "min_pct_7d": round(min_pct, 2) if min_pct else None,

            "close_7d": round(close_7, 2) if close_7 else None,
            "close_7d_pct": round(close7_pct, 2) if close7_pct else None,
        })

    return pd.DataFrame(results)

In [ ]:
seven_day_output = calculate_7_days_output(son_hali, df_hourly)

In [ ]:
print(seven_day_output.to_clipboard(index=False))